# TinyChirp SincNet-Time TensorFlow

Train a 1D CNN with a SincNet-style learnable frontend directly on raw audio waveforms, export an int8 TFLite model, and write a Rust `audio_sample.rs` file.

This mirrors `building_tensorflow/cnn_time.ipynb` but replaces the first convolutional layer with a custom Sinc-like frontend whose learned filters are later baked into a standard `Conv1D` layer for inference.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from typing import TYPE_CHECKING
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from building_tensorflow.utils import (  # noqa: E402
    SAMPLE_RATE,
    FRAME_LENGTH,
    FRAME_STEP,
    TARGET_FRAMES_TIME,
    TARGET_AUDIO_LEN_TIME,
    DATASET_ROOT,
    get_paths,
    configure_tf_runtime,
    set_global_seed,
    make_time_datasets,
)

if TYPE_CHECKING:
    import keras
else:
    from tensorflow import keras

configure_tf_runtime()
set_global_seed()

paths = get_paths("sincnet_time_tf")
OUT_TFLITE = paths.out_tflite
OUT_AUDIO_RS = paths.out_audio_rs

print("Dataset root:", DATASET_ROOT)
print("Model output:", OUT_TFLITE)
print("Audio sample output:", OUT_AUDIO_RS)
print("Sample rate:", SAMPLE_RATE)
print("Frame length:", FRAME_LENGTH)
print("Frame step:", FRAME_STEP)
print("Target frames (time):", TARGET_FRAMES_TIME)
print("Target audio length (time):", TARGET_AUDIO_LEN_TIME)

2026-04-10 11:45:35.613478: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-10 11:45:35.618314: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-10 11:45:35.648078: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-10 11:45:35.692418: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-10 11:45:35.742222: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

Dataset root: /home/nathan/Documents/tiny-chirp-microflow/dataset
Model output: /home/nathan/Documents/tiny-chirp-microflow/models/sincnet_time_tf.tflite
Audio sample output: /home/nathan/Documents/tiny-chirp-microflow/src/audio_sample.rs
Sample rate: 16000
Frame length: 1024
Frame step: 256
Target frames (time): 184
Target audio length (time): 47872


2026-04-10 11:45:38.290931: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-10 11:45:38.293430: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [3]:
train_ds, val_ds, test_ds, label_names = make_time_datasets()
num_labels = len(label_names)
print("Classes:", label_names)
print(len(train_ds),len(val_ds),len(test_ds))

Found 11292 files belonging to 2 classes.
Found 1380 files belonging to 2 classes.


2026-04-10 11:45:38.638716: I tensorflow_io/core/kernels/cpu_check.cc:128] Your CPU supports instructions that this TensorFlow IO binary was not compiled to use: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA


Found 1393 files belonging to 2 classes.
Classes: ['non_target' 'target']
353 44 44


## SincNet-style learnable frontend

We define a simplified SincNet-style learnable filterbank as a custom Keras layer that operates directly on the raw waveform. The layer maintains trainable parameters that are passed through a `sin` nonlinearity to produce filters, which are then applied via a 1D convolution.

In [4]:
import tensorflow as tf
import numpy as np
from building_tensorflow.utils import get_flops_native


class CustomFrontend(keras.layers.Layer):
    def __init__(self, num_filters: int, kernel_size: int, stride: int, **kwargs):
        super().__init__(**kwargs)
        self.num_filters = num_filters
        self.kernel_size = kernel_size
        self.stride = stride
        self.params = self.add_weight(
            shape=(kernel_size, 1, num_filters),
            initializer="random_normal",
            trainable=True,
            name="sinc_params",
        )

    def get_filters(self) -> tf.Tensor:
        return tf.math.sin(self.params)

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        return tf.nn.conv1d(inputs, self.get_filters(), stride=self.stride, padding="VALID")

In [9]:
POSSIBLE_FILTERS = [24, 32, 40]
POSSIBLE_KERNEL_SIZES = [32, 64, 128]
POSSIBLE_STRIDES = [16, 32, 64]
POSSIBLE_DENSE_HIDDEN = [48, 64, 80]

def build_training_model(num_labels: int, num_filters: int, kernel_size: int, stride: int, dense_hidden: int) -> "keras.Model":
    inputs = keras.Input(shape=(TARGET_AUDIO_LEN_TIME, 1))
    x = CustomFrontend(
        num_filters=num_filters, 
        kernel_size=kernel_size, 
        stride=stride, 
        name="sinc_frontend")(inputs)
    x = keras.layers.ReLU()(x)
    x = keras.layers.GlobalAveragePooling1D()(x)
    x = keras.layers.Dense(dense_hidden, activation="relu")(x)
    outputs = keras.layers.Dense(num_labels, activation=None)(x)
    return keras.Model(inputs, outputs, name="sincnet_time_training")

In [ ]:
from building_tensorflow.utils import evaluate_binary_classifier
def gather_results(num_filters: int, kernel_size: int, stride: int, dense_hidden: int):
    training_model = build_training_model(num_labels, num_filters, kernel_size, stride, dense_hidden)
    training_model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

    _= training_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=20,
        validation_steps=50,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True,
            )
        ],
    )

    metrics = evaluate_binary_classifier(training_model,
    train_ds,
    test_ds,
    threshold=0.5,)
    

    return metrics


In [16]:
from building_tensorflow.utils import EvalMetrics
import itertools
full_results: list[tuple[tuple[int, int, int, int], tuple[EvalMetrics, EvalMetrics]]] = []

for num_filters, kernel_size, stride, dense_hidden in itertools.product(POSSIBLE_FILTERS, POSSIBLE_KERNEL_SIZES, POSSIBLE_STRIDES, POSSIBLE_DENSE_HIDDEN):
    metrics = gather_results(num_filters, kernel_size, stride, dense_hidden)
    full_results.append(((num_filters, kernel_size, stride, dense_hidden), metrics))

full_results = sorted(full_results, key=lambda x: x[1][0].auc, reverse=True)


Epoch 1/20
316/353 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.6605 - loss: 0.6087

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 7))
for (num_filters, kernel_size, stride, dense_hidden), (train_m, _) in full_results:
    if train_m.roc_fpr is not None and train_m.roc_tpr is not None:
        ax.plot(
            train_m.roc_fpr,
            train_m.roc_tpr,
            alpha=0.5,
            linewidth=0.8,
            label=f"f={num_filters} k={kernel_size} s={stride} d={dense_hidden}  AUC={train_m.auc:.3f}",
        )

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_xscale("log")
ax.set_title("ROC curves — all hyperparameter combinations (train set)")
ax.legend(fontsize=5, loc="lower right", ncol=2)
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.show()

TOP_N = 10
header = f"{'Rank':<5} {'filters':<8} {'kernel':<8} {'stride':<8} {'dense':<8} {'value':<10} {'threshold':<10}"
sep = "-" * len(header)

for metric in ("accuracy", "precision", "recall", "f2"):
    print(f"\n=== Top {TOP_N} by test {metric} ===")
    ranked = sorted(full_results, key=lambda x, m=metric: getattr(x[1][1], m), reverse=True)[:TOP_N]
    print(header)
    print(sep)
    for rank, ((num_filters, kernel_size, stride, dense_hidden), (_, test_m)) in enumerate(ranked, 1):
        val = getattr(test_m, metric)
        print(f"{rank:<5} {num_filters:<8} {kernel_size:<8} {stride:<8} {dense_hidden:<8} {val:<10.4f} {test_m.threshold:<10.4f}")